# 01. Raw Data Ingestion and Telemetry Auditing
This notebook establishes raw dataset ingestion, schema validation, and null integrity across the 10 operational CSV feeds.


In [1]:
import os
import glob
from pathlib import Path
import pandas as pd

raw_dir = Path('../data/raw')
csv_files = sorted(list(raw_dir.glob('*.csv')))
datasets = {f.name: pd.read_csv(f) for f in csv_files}


## Dataset Manifest and Dimensional Inventory
Summary of row counts, column counts, and memory footprint across all ingested feeds.


In [2]:
manifest = []
for name, df in datasets.items():
    manifest.append({
        'File Name': name,
        'Rows': len(df),
        'Columns': len(df.columns),
        'Memory (KB)': round(df.memory_usage(deep=True).sum() / 1024, 2)
    })
pd.DataFrame(manifest)


,File Name,Rows,Columns,Memory (KB)
0,01_fund_master.csv,40,15,9.93
1,02_nav_history.csv,46000,3,1527.47
2,03_aum_by_fund_house.csv,90,5,5.98
3,04_monthly_sip_inflows.csv,48,6,2.71
4,05_category_inflows.csv,144,3,5.80
5,06_industry_folio_count.csv,21,6,1.26
6,07_scheme_performance.csv,40,19,9.20
7,08_investor_transactions.csv,32778,13,5500.45
8,09_portfolio_holdings.csv,322,8,33.57
9,10_benchmark_indices.csv,8050,3,350.52


## Missing Value and Null Distribution Analysis
Verifying zero unexpected missing values and quantifying operational nulls.


In [3]:
null_audit = []
for name, df in datasets.items():
    nulls = df.isnull().sum()
    null_cols = nulls[nulls > 0].to_dict()
    null_audit.append({
        'Dataset': name,
        'Has Nulls': bool(null_cols),
        'Null Details': str(null_cols) if null_cols else 'None'
    })
pd.DataFrame(null_audit)


,Dataset,Has Nulls,Null Details
0,01_fund_master.csv,False,None
1,02_nav_history.csv,False,None
2,03_aum_by_fund_house.csv,False,None
3,04_monthly_sip_inflows.csv,True,{'yoy_growth_pct': 12}
4,05_category_inflows.csv,False,None
5,06_industry_folio_count.csv,False,None
6,07_scheme_performance.csv,False,None
7,08_investor_transactions.csv,False,None
8,09_portfolio_holdings.csv,False,None
9,10_benchmark_indices.csv,False,None


## Key Identifier Uniqueness and Referential Baseline
Validating uniqueness of scheme codes, transaction records, and portfolio holdings.


In [4]:
assert datasets['01_fund_master.csv']['amfi_code'].is_unique
assert datasets['07_scheme_performance.csv']['amfi_code'].is_unique
print('AMFI primary key uniqueness verified across Fund Master and Scheme Performance.')


AMFI primary key uniqueness verified across Fund Master and Scheme Performance.
